In [18]:
from lignin_saf.ligsaf_chemicals import create_chemicals
from lignin_saf.ligsaf_settings import feed_parameters, prices
from lignin_saf.systems.rcf import create_rcf_system
from lignin_saf.systems.rcf_oil_purification import create_rcf_oil_purification_system
from lignin_saf.systems.monomer_purification import create_monomer_purification_system
from lignin_saf.systems.ligsaf_utilities import create_rcf_utilities_system
from lignin_saf.ligsaf_settings import hdo_params, h2_pressure
from lignin_saf.ligsaf_units import PSA

In [19]:
from biosteam import main_flowsheet as F
import biosteam as bst
import thermosteam as tmo
import pandas as pd
import numpy as np


In [20]:
# Code just to increase the number of display units for the various components
tmo.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
tmo.MultiStream.display_units.N = 40  
bst.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
bst.MultiStream.display_units.N = 40  

In [21]:

chems = create_chemicals()
bst.settings.set_thermo(chems)
bst.settings.CEPCI = 541.7   # 2016 USD basis

# Poplar group must be defined before creating any stream that references it
chems.define_group(
    name='Poplar',
    IDs=['Glucan', 'Xylan', 'Arabinan', 'Mannan', 'Galactan',
         'Sucrose', 'Lignin', 'Acetate', 'Extract', 'Ash'],
    composition=[0.464, 0.134, 0.002, 0.037, 0.014,
                 0.001, 0.285, 0.035, 0.016, 0.012],
    wt=True
)

poplar_in = bst.Stream('Poplar_In',
                       Poplar=feed_parameters['flow'] * 1e3,
                       Water=feed_parameters['moisture'] * feed_parameters['flow'] * 1e3,
                       phase='l', units='kg/d', price=prices['Feedstock'])

# ── Area 200: RCF process ──────────────────────────────────────────────────
rcf_system = create_rcf_system(ins=poplar_in)
rcf_system.simulate()

# ── Area 300: Purification ─────────────────────────────────────────────────
rcf_oil_purification_sys = create_rcf_oil_purification_system(ins=F.RCF_Oil)
monomer_purification_sys = create_monomer_purification_system(ins=F.Purified_RCF_Oil)
rcf_oil_purification_sys.simulate()
monomer_purification_sys.simulate()
BT, WWT, gas_mixer = create_rcf_utilities_system()

rcf_combined_system = bst.System(
    'Combined_RCF_System',
    path=(rcf_system, rcf_oil_purification_sys, monomer_purification_sys, WWT),
    facilities=[gas_mixer, BT],
)
rcf_combined_system.simulate()
rcf_combined_system.show()


c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: Poplar_In> has been replaced in registry
  self._register(ID)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_multi_stream.py:251: RuntimeWarning: <Stream: Meoh_recycle> has been replaced in registry
  self._register(ID)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <MultiStream: hydrogen_recycle> has been replaced in registry
  self._register(ID)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: Meoh_in> has been replaced in registry
  self._register(ID)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: Water_in_meoh> has been replaced in registry
  self._register(ID)
C:\Users\hwadg\AppData\Local\Temp\ipykernel_33292\3539105094.py:21: RuntimeWarning: <Mixer: MIX100> has been replaced in regist

System: Combined_RCF_System
Highest convergence error among components in recycle
streams {HX117-0, PUMP112-0} after 1 loops:
- flow rate   1.86e-10 kmol/hr (0.012%)
- temperature 1.84e-03 K (0.00059%)
ins...
[0] RCF_Catalyst  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): NiC  2.28
[1] air_lagoon  
    phase: 'g', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): N2  889
                    O2  219
[2] caustic  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water  33.4
                    NaOH   15.1
[3] Poplar_In  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water     925
                    Sucrose   0.243
                    Extract   7.4
                    Acetate   48.6
                    Ash       1e+03
                    Lignin    156
                    Glucan    238
                    Xylan     84.5
                    Arabinan  1.26
                    Mannan    19
                    Galactan  7.2
[4] Meoh_in  
    phase: 'l

In [22]:
# Different flows for the two monomers - strange, probably have to do with the scaling function used and the monomer yields. 
# This is why constantly checking the system is crucial

In [23]:
    # Hydrodeoxygenation reactions

hydrodeoxygenation_rxn = bst.ParallelReaction([
    bst.Reaction('Propylguaiacol,l + 6Hydrogen,g -> 1propylcyclohexane,l + 2Water,l + Methane,g', reactant='Propylguaiacol', phases='lg',
                    X=1.0, basis='mol'),
    bst.Reaction('1Propylsyringol,l + 8Hydrogen,g -> 1propylcyclohexane,l + 3Water,l + 2Methane,g', reactant='Propylsyringol', phases='lg',
                    X=1.0, basis='mol'),
])

In [65]:
h2_required = 6*(F.RCF_Monomers.imol['Propylguaiacol']) + 8*(F.RCF_Monomers.imol['Propylguaiacol'])
h2_required*1.5

365.3752699824305

In [ ]:
h2_in = bst.Stream(ID = 'Hydrogen_In', Hydrogen = 365.37 , units = 'kmol/hr', P = h2_pressure, phase = 'g') 
# Inlet hydrogen presure


c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: Hydrogen_In> has been replaced in registry
  self._register(ID)


In [25]:
dodcane_required = hdo_params['solvent_req']   #m3/kg of lignin oil
dodecane_flow = F.RCF_Monomers.F_mass * dodcane_required
solvent_stream = bst.Stream(ID = 'Dodecane_In', Dodecane = dodecane_flow, units = 'm3/hr', P = 101325, T = 300, phase = 'l')

c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: Dodecane_In> has been replaced in registry
  self._register(ID)


In [26]:
catalyst_required = hdo_params['catalyst_req']      # kg/kg of lignin oil
catalyst_flow = F.RCF_Monomers.F_mass * catalyst_required
catalyst_stream =  bst.Stream(ID = 'HDO_Catalyst_In', Ni2PSiO2 = catalyst_flow, units = 'kg/hr', phase = 's')


c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: HDO_Catalyst_In> has been replaced in registry
  self._register(ID)


In [27]:
mixer = bst.units.Mixer(ins = (h2_in, F.RCF_Monomers, solvent_stream), rigorous = True)
mixer.simulate()


In [28]:
compressor = bst.units.IsentropicCompressor(ins = mixer-0, P = hdo_params['P'], vle = True)
compressor.simulate()

In [29]:
heater = bst.units.HXutility(ins = compressor-0, T = hdo_params['T'], rigorous= True)
heater.simulate()

In [30]:
import biosteam as bst, numpy as np
from math import ceil

from typing import Optional
from biosteam.units.design_tools import (
    PressureVessel, 
)
 

class HydrodeoxygenationReactor(bst.Unit, bst.units.design_tools.PressureVessel):

    """
    Batch reactor for HDO of lignin oil
    Designed to process RCF monomers only right now, but functionality for dimers needs to be added

    Reaction based on ring hydrogenation + aryl bond cleavage to produce cycloalkanes
    Alternative reaction scheme, where aromatic ring of monomers is preserved and just the aryl bond is cleaved (CBI) might be considered later

    Duty estimated based on ring hydrogenation enthalpy stated in [3]. Could be refined 

    All reaction conditions are typically cited from [1],[2] unless noted otherwise   

    Maximum volume is capped at 600 m3 [4], and number of reactors is scaled accordingly. Hence the reactors are technically oversized

    
    References
    ----------------------------------------------------------------------------------
        [1] Bruno Pandalone et al.,
        "Optimum Lignin Oil - Finding the Most Suitable Feedstock to Replace Cycloalkanes in Sustainable Aviation Fuel (SAF)"
        ChemSusChem. 2025. 18(11). https://doi.org/10.1002/cssc.202402531
        [2] Bruno Pandalone et al.,
        "Molecule-to-molecule conversion of RCF lignin oil to sustainable aviation fuel"
        Chem Circularity. 2026. https://doi.org/10.1016/j.checir.2026.100025
        [3] Matthew S. Webber et al., 
        " Lignin deoxygenation for the production of sustainable aviation fuel blendstocks"
        Nature Materials. 2024. 23, 1622-1638. https://doi.org/10.1038/s41563-024-02024-6
        [4] Andrew W. Bartling, et al.,
        "Techno-economic analysis and life cycle assessment of a biorefinery utilizing reductive catalytic fractionation." 
        Energy & Environmental Science. 2021. 4147-4168. https://doi.org/10.1039/D1EE01642C
        [5] Zhengwen Cao et al.,
        "A Convergent Approach for a Deep Converting Lignin-First Biorefinery Rendering High-Energy-Density Drop-in Fuels."
        Joule. 2018. 2(6). https://doi.org/10.1016/j.joule.2018.03.012
    -----------------------------------------------------------------------------------

    """



    _F_BM_default = {'Horizontal pressure vessel': 3.05,
                     'Vertical pressure vessel': 4.16,
                     'Platform and ladders': 1.}                   

    _N_ins = 2
    _N_outs = 2
    
    _units = {**PressureVessel._units,
              'Batch time': 'hr',
              'Turnaround time': 'hr',
              'Time on stream': 'hr',
              'Total beds': "",
              'Beds in service': "",
              'Total volume': 'm3',
              'Reactor volume': 'm3',
              'Duty' : 'kJ/hr'}
    


    # Default operating temperature [K]
    T_default: float = 573.15                       # 300 C from [1][2][5]

    #: Default operating pressure [Pa]
    P_default:  float = 5e6                         # 5 MPa from [1][2][5]
    
    #: Default reaction time [hr]
    tau_default: float = 5                          # Total 5 hr reaction time [1][2]

    #: Default cleaning and unloading time (hr).
    tau_0_default: float  = 1                       # Assumed time required to cool down the reactor 

    # Fixed bed configuration
    N_total_default: int =  3                       # Total beds (2 operating + 1 cleaning)

    N_working_default: int = 2                      # Beds operating at any time

    # Default free-space fraction of reactor volume
    free_frac_default: float = 0.10                 # 10% kept free for gas disengagement / headspace
    
    # Default maximum vessel volume [m3]
    V_max_default: float = 600                     # Assumed, as was maximum volume in [4]

    # Aspect ratio (L/D of the reactor)
    aspect_ratio: float = 5.0                      # Assumed



    def _init(
            self,
            T: Optional[float] = None,
            P: Optional[float] = None,
            tau: Optional[float] = None,
            vessel_material: Optional[str] = None,
            vessel_type: Optional[str] = None,
            tau_0: Optional[float] = None,
            free_frac: Optional[float] = None,
            V_max: Optional[float] = None,
            aspect_ratio: Optional[float] = None,
            *,
            reaction_1            
            ):


        self.T = self.T_default if T is None else T
        self.P = self.P_default if P is None else P
        self.tau = self.tau_default if tau is None else tau
        self.vessel_material = 'Stainless steel 316' if vessel_material is None else vessel_material
        self.vessel_type = 'Vertical' if vessel_type is None else vessel_type
        self.tau_0 = self.tau_0_default if tau_0 is None else tau_0
        self.free_frac      = self.free_frac_default      if free_frac      is None else free_frac
        self.V_max = self.V_max_default if V_max is None else V_max
        self.aspect_ratio          = self.aspect_ratio         if aspect_ratio          is None else aspect_ratio
        self.reaction = reaction_1
        # heat_exchanger_1 = self.auxiliary('heat_exchanger_1', bst.HXutility, pump_1.outs[0])






    def _size_bed(self):

        cycle_time        = self.tau + self.tau_0                  

        # Total monomer flow
        total_monomer_flow = self.ins[0].F_vol                      # [m3/hr]

        V_theoretical = total_monomer_flow * self.tau               # [m3] Theoretical volume required
        
        V_actual = V_theoretical*(1+self.free_frac)                 # [m3] Actual volume required based on free fraction
        
        N_working = ceil(V_actual/self.V_max)                       # Number of working reactors based off maximum volume
        N_offline = ceil(N_working*(self.tau_0/cycle_time))         # Number of offline beds, calculated based off cleaning time and the total cycle time, rounded off to the next number
        N_total = N_working + N_offline                             # Total beds required
        V_total = N_total * self.V_max                              # Total volume required
        
        diameter = ((4*self.V_max)/(self.aspect_ratio*np.pi))**(1/3)
        length = self.aspect_ratio * diameter


        return length, diameter, N_total, N_working, V_total

        
    def _run(self):
        inf, catalyst_in = self.ins
        eff, catalyst_out = self.outs

        eff.copy_like(inf)
        self.reaction(eff)

        eff.imass['l', 'Dodecane'] = eff.imass['l', 'Dodecane']*(1-hdo_params['solvent_decomp']) # 0.5% dodecane lost due to decomposition
        eff.imass['g', 'Dodecane'] = eff.imass['g', 'Dodecane']*(1-hdo_params['solvent_decomp']) # 0.5% dodecane lost due to decomposition
        
        catalyst_out.copy_like(catalyst_in)


        eff.P = self.P                                             # Assuming no P drop sinnce its a batch reactor
        eff.T = self.T                                             # Assuming isothermal operation
    
        

    def _design(self):
        length, diameter, N_total, N_working, V_total = self._size_bed()   # Calling size bed function to determine diameter and length 
        
        cycle_time = self.tau + self.tau_0
        self.set_design_result('Diameter', 'ft', diameter)
        self.set_design_result('Length', 'ft', length)
        self.set_design_result('Reactor volume', 'm3', self.V_max)
        self.set_design_result('Total volume', 'm3', V_total)
        self.set_design_result('Total beds', '', N_total)
        self.set_design_result('Beds in service', '', N_working)
        self.set_design_result('Time on stream', 'hr', self.tau)
        self.set_design_result('Turnaround time', 'hr', self.tau_0)
        self.set_design_result('Batch time', 'hr', cycle_time)


        
        
        # Calculates weight based off pressure, diameter and length
        # Adds vessel type wall thickness, vessel weight, diameter and length to dictionary
        # But diameter and length are already there because of set_design_result above
        
        self.design_results.update(
            self._vertical_vessel_design(    
                self.P*(1/6894.76),
                self.design_results['Diameter']*3.28084,
                self.design_results['Length']*3.28084
            )
        )
        
                            

        duty = -70 * self.ins[0].imol['Hydrogen'] * 1000  # Aromatic hydrogenation has duty between 58-70 kJ/mol H2 according to https://www.nature.com/articles/s41563-024-02024-6

        self.add_heat_utility(duty/N_total, self.T, agent = bst.HeatUtility.get_cooling_agent('chilled_water'))  # BioSTEAM defaulted to cooling water but chilled water more probable since reaction is very exothermic 
        self.set_design_result('Duty', 'kJ/hr', duty)





    def _cost(self):
        design = self.design_results # Calling the dictionary used to store design results in design method above 

        baseline_purchase_costs = self.baseline_purchase_costs # Dictionary for storing baseline costs

        weight = design['Weight']  # weight parameter stores the value from the 'Weight' key in the design dictionnary
        
        N_reactors = design['Total beds']
        # Calculates the baseline purchase cost based off diameter length and weight
        baseline_purchase_costs.update( 
            self._vessel_purchase_cost(weight, design['Diameter'], design['Length'])
        )

        self.parallel['self'] = N_reactors # Used to create multiple of the same beds


        
       

    

In [31]:
tau = 2
tau_0 = 1
free_frac = 0.1
cycle_time        = tau + tau_0                  
V_max = 600
aspect_ratio = 5



# Total monomer flow
total_monomer_flow = compressor.outs[0].F_vol                      # [m3/hr]

V_theoretical = total_monomer_flow * tau               # [m3] Theoretical volume required

V_actual = V_theoretical*(1+free_frac)                 # [m3] Actual volume required based on free fraction

N_working = ceil(V_actual/V_max)                       # Number of working reactors based off maximum volume
N_offline = ceil(N_working*(tau_0/cycle_time))         # Number of offline beds, calculated based off cleaning time and the total cycle time, rounded off to the next number
N_total = N_working + N_offline                             # Total beds required
V_total = N_total * V_max                              # Total volume required

diameter = ((4*V_max)/(aspect_ratio*np.pi))**1/3
length = aspect_ratio * diameter


In [32]:
HDO = HydrodeoxygenationReactor(ins = (heater-0, catalyst_stream),
                                T = hdo_params['T'],
                                P = hdo_params['P'],
                                tau = hdo_params['tau'],
                                tau_0 = hdo_params['tau_0'],
                                free_frac = hdo_params['free_frac'],
                                V_max = hdo_params['V_max'],
                                aspect_ratio = hdo_params['aspect_ratio'],
                                reaction_1 = hydrodeoxygenation_rxn)
HDO.simulate()

C:\Users\hwadg\AppData\Local\Temp\ipykernel_33292\1209509954.py:192: CostWarning: <HydrodeoxygenationReactor: R2> Vertical vessel weight (1.646e+06 lb) is out of bounds (4200 to 1e+06 lb) for cost correlation
  self._vertical_vessel_design(
C:\Users\hwadg\AppData\Local\Temp\ipykernel_33292\1209509954.py:192: CostWarning: <HydrodeoxygenationReactor: R2> Vertical vessel length (87.7 ft) is out of bounds (12 to 40 ft) for cost correlation
  self._vertical_vessel_design(


In [33]:
HDO

HydrodeoxygenationReactor: R2
ins...
[0] s95  from  HXutility-H2
    phases: ('g', 'l'), T: 573.15 K, P: 5e+06 Pa
    flow (kmol/hr): (g) Hydrogen        300
                        Dodecane        35.5
                    (l) Hydrogen        5.12e-12
                        Dodecane        976
                        Propylguaiacol  17.4
                        Propylsyringol  14.7
[1] HDO_Catalyst_In  
    phase: 's', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Ni2PSiO2  4.63e+03
outs...
[0] s97  
    phases: ('g', 'l'), T: 573.15 K, P: 5e+06 Pa
    flow (kmol/hr): (g) Hydrogen           77.7
                        Methane            46.9
                        Dodecane           33.7
                    (l) Water              79
                        Hydrogen           5.12e-12
                        propylcyclohexane  32.1
                        Dodecane           927
[1] s98  
    phase: 's', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Ni2PSiO2  4.63e+03


In [34]:
HDO.results()

Hydrodeoxygenation reactor                           Units        R2
Chilled water       Duty                             kJ/hr  -2.1e+07
                    Flow                           kmol/hr  1.39e+04
                    Cost                            USD/hr       105
Design              Diameter                            ft      17.5
                    Length                              ft      87.7
                    Reactor volume                      m3       600
                    Total volume                        m3      4200
                    Total beds                                     7
                    Beds in service                                6
                    Time on stream                      hr         5
                    Turnaround time                     hr         1
                    Batch time                          hr         6
                    Vessel type                             Vertical
                    Weight                              lb  1.65e+06
                    Wall thickness                      in      6.82
                    Duty                             kJ/hr  -2.1e+07
Purchase cost       Vertical pressure vessel (x7)      USD  2.67e+07
                    Platform and ladders (x7)          USD  5.39e+05
Total purchase cost                                    USD  2.73e+07
Utility cost                                        USD/hr       105

In [35]:
HDO

HydrodeoxygenationReactor: R2
ins...
[0] s95  from  HXutility-H2
    phases: ('g', 'l'), T: 573.15 K, P: 5e+06 Pa
    flow (kmol/hr): (g) Hydrogen        300
                        Dodecane        35.5
                    (l) Hydrogen        5.12e-12
                        Dodecane        976
                        Propylguaiacol  17.4
                        Propylsyringol  14.7
[1] HDO_Catalyst_In  
    phase: 's', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Ni2PSiO2  4.63e+03
outs...
[0] s97  
    phases: ('g', 'l'), T: 573.15 K, P: 5e+06 Pa
    flow (kmol/hr): (g) Hydrogen           77.7
                        Methane            46.9
                        Dodecane           33.7
                    (l) Water              79
                        Hydrogen           5.12e-12
                        propylcyclohexane  32.1
                        Dodecane           927
[1] s98  
    phase: 's', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Ni2PSiO2  4.63e+03


In [36]:
# Cooling the batch reactor before getting the contents out
cooler = bst.units.HXutility(ins = HDO-0, T = 400, rigorous = True)
cooler.simulate()

c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Methane has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)


In [37]:
# Depressurizing for safety
valve = bst.units.IsenthalpicValve(ins = cooler-0, P = 101325, vle = True)
valve.simulate()

In [38]:
# Flashing the gaseous components
flash = bst.units.Flash(ins = valve-0, T = 298, P = 101325)
flash.simulate()

In [39]:
# Another flash downstream to remove water before PSA
# The liquid outlet is almost pure water and has very little levels of methane, have to think about where to outlet this 
flash_2 = bst.units.Flash(ins = flash-0, T = 275, P = 5e5)
flash_2.simulate()

c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\design_tools\pressure_vessel.py:104: CostWarning: <Flash: F2> Vertical vessel weight (362.8 lb) is out of bounds (4200 to 1e+06 lb) for cost correlation
  return method(pressure, diameter, length)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\design_tools\pressure_vessel.py:104: CostWarning: <Flash: F2> Vertical vessel length (6.5 ft) is out of bounds (12 to 40 ft) for cost correlation
  return method(pressure, diameter, length)


In [40]:
# I believe 303 is a requirement for PSA but I'll need to double check
psa_heater = bst.units.HXutility( ins=flash_2-0, T=303, rigorous=True)
psa_heater.simulate()

In [41]:
# PSA for HDO - the hdo_gases will need to be routed to BT gas feed for combustion
psa_hdo = PSA(ins=psa_heater-0, outs=('', 'hdo_gases')) 
psa_hdo.simulate()

c:\users\hwadg\onedrive - the pennsylvania state university\shi_wadgama_shared\models\atjspk\lignin_saf\ligsaf_units.py:956: CostWarning: <PSA: U1> Vertical vessel weight (83.73 lb) is out of bounds (4200 to 1e+06 lb) for cost correlation
  self._vertical_vessel_design(
c:\users\hwadg\onedrive - the pennsylvania state university\shi_wadgama_shared\models\atjspk\lignin_saf\ligsaf_units.py:956: CostWarning: <PSA: U1> Vertical vessel length (2.01 ft) is out of bounds (12 to 40 ft) for cost correlation
  self._vertical_vessel_design(
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\_unit.py:986: RuntimeWarning: the purchase cost item, 'Adsorbent cost', has no defined bare-module factor in the 'PSA.F_BM' dictionary; bare-module factor now has a default value of 1
  warn(f"the purchase cost item, '{name}', has "


In [42]:
flash_2-1

Stream: s104 from <Flash: F2>
phase: 'l', T: 275 K, P: 500000 Pa
flow (kmol/hr): Water              22.3
                Hydrogen           3.3e-10
                Methane            0.0306
                propylcyclohexane  0.000599
                Dodecane           5.5e-05


In [43]:
hdo_solvent_recovery = bst.units.BinaryDistillation(ins=flash-1,
    LHK=('propylcyclohexane', 'Dodecane'),
    Lr=0.99, Hr=1 - 0.0001, P=101325,
    vessel_material='Stainless steel 316',
    k=2, partial_condenser=True,
)
hdo_solvent_recovery.simulate()

In [44]:
hdo_solvent_recovery

BinaryDistillation: D1
ins...
[0] s102  from  Flash-F1
    phase: 'l', T: 298 K, P: 101325 Pa
    flow (kmol/hr): Water              56.5
                    Hydrogen           8.42e-10
                    Methane            0.0892
                    propylcyclohexane  32.1
                    Dodecane           961
outs...
[0] s108  
    phase: 'g', T: 394.48 K, P: 101325 Pa
    flow (kmol/hr): Water              56.5
                    Hydrogen           8.42e-10
                    Methane            0.0892
                    propylcyclohexane  31.8
                    Dodecane           0.0961
[1] s109  
    phase: 'l', T: 489.41 K, P: 101325 Pa
    flow (kmol/hr): propylcyclohexane  0.321
                    Dodecane           960


In [45]:
hdo_product_recovery = bst.units.BinaryDistillation(
    ins=hdo_solvent_recovery-0,
    outs = ('HDO_WW', 'SAF_CycloAlkane'),
    LHK=('Water', 'Propylcyclohexane'),
    y_top=0.9, x_bot=0.001, P=101325, k=2,
)
hdo_product_recovery.simulate()

c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\dew_point.py:129: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\dew_point.py:129: RuntimeWarning: Methane has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)


In [46]:
hdo_product_recovery

BinaryDistillation: D2
ins...
[0] s108  from  BinaryDistillation-D1
    phase: 'g', T: 394.48 K, P: 101325 Pa
    flow (kmol/hr): Water              56.5
                    Hydrogen           8.42e-10
                    Methane            0.0892
                    propylcyclohexane  31.8
                    Dodecane           0.0961
outs...
[0] HDO_WW  
    phase: 'g', T: 357.54 K, P: 101325 Pa
    flow (kmol/hr): Water              56.5
                    Hydrogen           8.42e-10
                    Methane            0.0892
                    propylcyclohexane  6.28
[1] SAF_CycloAlkane  
    phase: 'l', T: 426.37 K, P: 101325 Pa
    flow (kmol/hr): Water              0.0255
                    propylcyclohexane  25.5
                    Dodecane           0.0961


In [47]:
# Trying to create a mixer that can actually mix the catalyst without changing its phase to liquid
class HDOMixer(bst.Unit):

    _N_ins = 2
    _N_outs = 1

    def _init(self):
        ins, catalyst = self.ins
        outs, = self.outs
        outs.copy_like(ins)
        outs.imass['s','Ni2PSiO2'] = catalyst.imass['s','Ni2PSiO2']

    
    def _run(self):
        pass

    def _design(self):
        pass


In [ ]:
mixer_2 = bst.units.Mixer(ins = (HDO-1, valve-0), rigorous  = True)
mixer_2.simulate()